# Homework assignment: Object detection

This notebook completes the object detection homework using OpenCV, Matplotlib, and a pretrained YOLOv8 model from Ultralytics. The target image path is set to `../data/soccer/brazil-vs-croatia-neymar.jpg`, as required by the prompt.

## Task 0: Install and verify Ultralytics

The assignment asks us to install the Ultralytics package using either pip or conda and then verify the installation by importing the library and printing its version. In a notebook, the pip command can be run directly with `!pip install ultralytics`.

In [ ]:
# If ultralytics is not installed, uncomment and run one of the following commands.
# !pip install ultralytics
# !conda install -c conda-forge ultralytics -y

In [ ]:
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## Task 1: Load and display the image using OpenCV and Matplotlib

OpenCV loads images in BGR color format by default, while Matplotlib expects RGB format. Therefore, after reading the image with `cv2.imread()`, we convert it from BGR to RGB before displaying it.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

image_path = Path('../data/soccer/brazil-vs-croatia-neymar.jpg')

if not image_path.exists():
    raise FileNotFoundError(
        f'Image not found at {image_path}. Make sure the notebook is run from the expected folder or update image_path.'
    )

img_bgr = cv2.imread(str(image_path))

if img_bgr is None:
    raise ValueError('cv2.imread() failed. Check that the file path is correct and the image file is readable.')

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.title('Soccer Image')
plt.show()

print('Image path:', image_path)
print('BGR image shape:', img_bgr.shape)
print('RGB image shape:', img_rgb.shape)

### Interpretation for Task 1

The image was successfully loaded using OpenCV and displayed using Matplotlib. The conversion from BGR to RGB is necessary because OpenCV and Matplotlib use different default channel orders. Without this conversion, the displayed colors would be incorrect.

## Task 2: Load YOLOv8 and run object detection

We load a pretrained YOLOv8 nano model using the `yolov8n.pt` weights. The model is applied directly to the original OpenCV BGR image, as required by the prompt. The prediction uses `save=False` and `imgsz=640`.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.predict(
    source=img_bgr,
    save=False,
    imgsz=640
)

# Display the first result using Ultralytics' built-in display method.
results[0].show()

In [ ]:
# Alternative notebook-friendly display of the annotated result.
annotated_bgr = results[0].plot()
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(annotated_rgb)
plt.axis('off')
plt.title('YOLOv8 Object Detection Results')
plt.show()

### Interpretation for Task 2

The pretrained YOLOv8 model detects objects in the soccer image and returns bounding boxes, class IDs, confidence scores, and class names. Since the model is pretrained on the COCO dataset, it can detect common categories such as people and sports balls without additional training on this specific soccer image.

## Task 3: Access and print detection results

The detection results are stored in `results[0].boxes`. For each detected box, we extract the class ID, confidence score, and bounding box coordinates. The coordinates are printed in `[x1, y1, x2, y2]` format.

In [ ]:
boxes = results[0].boxes

print('Number of detections:', len(boxes))

for i, box in enumerate(boxes):
    class_id = int(box.cls.cpu().item())
    confidence = float(box.conf.cpu().item())
    coordinates = box.xyxy.cpu().numpy().flatten()
    class_name = model.names[class_id]

    print(f'Detection {i + 1}')
    print(f'  Class ID: {class_id}')
    print(f'  Class name: {class_name}')
    print(f'  Confidence: {confidence:.2f}')
    print(f'  Bounding box [x1, y1, x2, y2]: {coordinates}')
    print()

### Interpretation for Task 3

Each detection consists of a predicted class, a confidence score, and a bounding box. The confidence score measures how certain the model is about the detected object class, while the bounding box coordinates specify the object's location in the image. This output is the core result of object detection: both classification and localization are performed at the same time.

## Task 4: Print class IDs and names for `person` and `sports ball`

The YOLO model stores a mapping from class IDs to class names in `model.names`. We loop through this dictionary and print the IDs for the categories `person` and `sports ball`.

In [ ]:
target_classes = {'person', 'sports ball'}

for class_id, class_name in model.names.items():
    if class_name in target_classes:
        print(f'Class ID: {class_id}, Class name: {class_name}')

### Interpretation for Task 4

The `person` and `sports ball` categories are especially relevant for a soccer image because the main objects of interest are soccer players and the ball. Identifying their class IDs makes it easier to filter detections later if we only want soccer-related objects.

## Optional extension: Filter and display only soccer-relevant detections

This section is not strictly required, but it is useful for interpreting the soccer-player image. We filter detections to keep only `person` and `sports ball` objects.

In [ ]:
person_id = [k for k, v in model.names.items() if v == 'person'][0]
sports_ball_id = [k for k, v in model.names.items() if v == 'sports ball'][0]

soccer_relevant = []

for i, box in enumerate(results[0].boxes):
    class_id = int(box.cls.cpu().item())
    if class_id in [person_id, sports_ball_id]:
        soccer_relevant.append({
            'detection_index': i + 1,
            'class_id': class_id,
            'class_name': model.names[class_id],
            'confidence': float(box.conf.cpu().item()),
            'bbox_xyxy': box.xyxy.cpu().numpy().flatten().tolist()
        })

soccer_relevant

## Final conclusion

This notebook demonstrates how to use a pretrained YOLOv8 model for object detection. Unlike image classification, object detection identifies both what objects are present and where they are located. In the soccer image, the most important classes are `person` and `sports ball`, which correspond to players and the soccer ball. The YOLOv8 model can detect these objects without task-specific training because it has already learned general object detection patterns from a large pretrained dataset.